# 05 — Backtest Validation
Runs a deterministic bar-by-bar backtest of the trained **DeepScalper** models on the
held-out validation portion of each ticker's data (last 20% — same split as `03_train_deepscalper.ipynb`).

**How it works:**
- Loads the raw `.parquet` bar data from Drive and recomputes features using the shared pipeline.
- Uses the **same 80/20 time-ordered split** as training — the model never saw the val portion.
- For each val bar, builds the Dict observation `{lob, priv, macro}` and runs **greedy** (ε=0) BDQ inference.
- Only the **direction branch** (`q_dir`) drives trading: HOLD / BUY / SELL.
- Tracks portfolio value bar-by-bar (1-min resolution) with 0.1% round-trip transaction costs.

**Output:**
- Per-ticker metrics table: total return, annualised Sharpe, max drawdown, win rate, trade count.
- Aggregate equal-weight equity curve vs. breakeven.

**Input:**  `/content/drive/MyDrive/algo_trader/weights/{TICKER}.pth`  
           `/content/drive/MyDrive/algo_trader/data/raw/{TICKER}.parquet`

In [ ]:
!pip install -q torch pyarrow pandas numpy matplotlib tqdm pytz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

WEIGHTS_DIR = '/content/drive/MyDrive/algo_trader/weights'
RAW_DIR     = '/content/drive/MyDrive/algo_trader/data/raw'
REPO_DIR    = '/content/deepscalper_copilot'

print(f'Weights dir : {WEIGHTS_DIR}')
print(f'Raw data dir: {RAW_DIR}')

In [ ]:
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'

if not os.path.exists(REPO_DIR + '/.git'):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

# algo_trader on sys.path → enables `from colab.deepscalper.X import Y`
ALGO_DIR = REPO_DIR + '/algo_trader'
if ALGO_DIR not in sys.path:
    sys.path.insert(0, ALGO_DIR)

print('Repo on path ✓')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from pathlib import Path
from tqdm.notebook import tqdm

from colab.deepscalper.architecture import DeepScalperNet
from colab.deepscalper.utils import (
    compute_macro_features,
    compute_micro_features,
    compute_day_starts,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')

# ── Architecture — must match config.py and training ─────────────────────────
MACRO_DIM     = 11
LOB_DIM       = 5
PRIV_DIM      = 2
N_DIR         = 3
N_SIZE        = 4
GRU_HIDDEN    = 128
MACRO_EMBED   = 64
FC_HIDDEN     = 128
LOOKBACK_BARS = 60

# ── Data split — must match 03_train_deepscalper.ipynb ───────────────────────
TRAIN_FRAC    = 0.80
TC_PCT        = 0.001    # One-way transaction cost (0.1 %)

SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]
print(f'{len(SP100_TICKERS)} tickers loaded')

In [ ]:
# ── Backtest engine ───────────────────────────────────────────────────────────

_DIR_HOLD = 0
_DIR_BUY  = 1
_DIR_SELL = 2


def run_backtest(
    model:          DeepScalperNet,
    macro_feats:    np.ndarray,
    lob_feats:      np.ndarray,
    close_arr:      np.ndarray,
    val_day_starts: list,
    lookback:       int   = 60,
    tc_pct:         float = 0.001,
    device:         str   = 'cpu',
) -> dict:
    """Deterministic greedy bar-by-bar backtest on the held-out val set.

    Runs every val day in calendar order (not randomly), resets position and
    private state at each day open (intraday-only strategy), and forces any
    open position to close at the day's final bar.

    Args:
        model          : DeepScalperNet in eval() mode.
        macro_feats    : (n_val_bars, 11) macro features for the val slice.
        lob_feats      : (n_val_bars,  5) micro features for the val slice.
        close_arr      : (n_val_bars,)    close prices for the val slice.
        val_day_starts : Bar indices (relative to val slice) where each day starts.
        lookback       : Observation window length (bars).
        tc_pct         : One-way transaction cost as fraction of trade value.
        device         : Torch device string.

    Returns:
        Dict with: equity_curve (list[float]), total_return, sharpe,
                   max_drawdown, win_rate, n_trades.
    """
    model.eval()
    n_bars      = len(close_arr)
    portfolio   = 1.0
    equity_curve = [1.0]
    step_returns = []
    trade_pnls   = []

    for day_idx, day_start in enumerate(val_day_starts):
        day_end = (
            val_day_starts[day_idx + 1] - 1
            if day_idx + 1 < len(val_day_starts)
            else n_bars - 1
        )

        # Reset intraday state
        position     = 0
        entry_price  = 0.0
        priv_history = deque(
            [np.zeros(2, dtype=np.float32)] * lookback, maxlen=lookback
        )

        t_start = day_start + lookback - 1
        if t_start >= day_end:
            continue  # Day too short for a full observation window

        for t in range(t_start, day_end):
            # ── Build observation ─────────────────────────────────────────
            win_start = max(0, t - lookback + 1)
            lob_seq   = lob_feats[win_start: t + 1]
            if len(lob_seq) < lookback:
                pad     = np.zeros((lookback - len(lob_seq), lob_feats.shape[1]), dtype=np.float32)
                lob_seq = np.vstack([pad, lob_seq])

            priv_arr = np.array(list(priv_history), dtype=np.float32)  # (lookback, 2)
            macro    = macro_feats[t]                                   # (11,)

            # ── Greedy inference ──────────────────────────────────────────
            with torch.no_grad():
                lob_t  = torch.tensor(lob_seq[None],  dtype=torch.float32, device=device)
                priv_t = torch.tensor(priv_arr[None],  dtype=torch.float32, device=device)
                mac_t  = torch.tensor(macro[None],     dtype=torch.float32, device=device)
                q_dir, _, _ = model(lob_t, priv_t, mac_t)

            dir_act = int(q_dir.argmax(1).item())

            current_price = float(close_arr[t])
            next_price    = float(close_arr[min(t + 1, day_end)])
            tc_cost       = 0.0

            # ── Execute direction action ──────────────────────────────────
            if dir_act == _DIR_BUY and position <= 0:
                if position < 0:  # Close short
                    pnl = (entry_price - current_price) / (entry_price + 1e-10)
                    trade_pnls.append(pnl)
                    tc_cost += tc_pct
                position    = 1
                entry_price = current_price
                tc_cost    += tc_pct

            elif dir_act == _DIR_SELL and position >= 0:
                if position > 0:  # Close long
                    pnl = (current_price - entry_price) / (entry_price + 1e-10)
                    trade_pnls.append(pnl)
                    tc_cost += tc_pct
                position    = -1
                entry_price = current_price
                tc_cost    += tc_pct
            # else: HOLD — no trade

            # ── Step return (log-return × position) ──────────────────────
            if position != 0 and current_price > 0:
                log_ret = float(np.log(next_price / (current_price + 1e-10)))
                step_r  = log_ret * position - tc_cost
            else:
                step_r = -tc_cost

            step_returns.append(step_r)
            portfolio    *= float(np.exp(step_r))
            equity_curve.append(portfolio)

            # ── Update private state ──────────────────────────────────────
            unreal_pnl = (
                (current_price - entry_price) / (entry_price + 1e-10) * position
                if position != 0 else 0.0
            )
            priv_history.append(np.array(
                [1.0 if position != 0 else 0.0,
                 float(np.clip(unreal_pnl, -0.5, 0.5))],
                dtype=np.float32,
            ))

        # ── Force-close any open position at EOD ──────────────────────────
        if position != 0 and entry_price > 0 and day_end < n_bars:
            eod_price = float(close_arr[day_end])
            pnl = (
                (eod_price - entry_price) / (entry_price + 1e-10) * position
            )
            trade_pnls.append(pnl)

    # ── Aggregate metrics ─────────────────────────────────────────────────────
    total_return = portfolio - 1.0

    arr    = np.array(step_returns, dtype=np.float64)
    sharpe = (
        float(arr.mean() / (arr.std() + 1e-10)) * np.sqrt(252 * 390)
        if len(arr) > 1 else 0.0
    )

    eq      = np.array(equity_curve, dtype=np.float64)
    peak    = np.maximum.accumulate(eq)
    max_dd  = float(((eq - peak) / (peak + 1e-10)).min())

    win_rate = float(np.mean([p > 0 for p in trade_pnls])) if trade_pnls else 0.0
    n_trades = len(trade_pnls)

    return {
        'equity_curve': equity_curve,
        'total_return': total_return,
        'sharpe':       sharpe,
        'max_drawdown': max_dd,
        'win_rate':     win_rate,
        'n_trades':     n_trades,
    }


# ── Main backtest loop ────────────────────────────────────────────────────────
all_results   = {}
equity_curves = {}

for ticker in tqdm(SP100_TICKERS, desc='Backtesting'):
    wt_path  = Path(WEIGHTS_DIR) / f'{ticker}.pth'
    raw_path = Path(RAW_DIR)     / f'{ticker}.parquet'

    if not wt_path.exists():
        print(f'{ticker}: no weights — skipping.')
        continue
    if not raw_path.exists():
        print(f'{ticker}: no raw data — skipping.')
        continue

    # ── Load & compute features ───────────────────────────────────────────────
    bars = pd.read_parquet(str(raw_path))
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    macro_feats = compute_macro_features(bars)
    lob_feats   = compute_micro_features(bars)
    close_arr   = bars['close'].values.astype(np.float64)
    day_starts  = compute_day_starts(bars.index)

    # ── Same 80/20 split as training ─────────────────────────────────────────
    n_bars    = len(bars)
    split_bar = int(n_bars * TRAIN_FRAC)
    val_day_starts = [d - split_bar for d in day_starts if d >= split_bar]

    if len(val_day_starts) < 2:
        print(f'{ticker}: insufficient val days — skipping.')
        continue

    # ── Load checkpoint ───────────────────────────────────────────────────────
    try:
        ckpt       = torch.load(str(wt_path), map_location=DEVICE, weights_only=True)
        state_dict = ckpt.get('online_net', ckpt)
        model = DeepScalperNet(
            macro_dim   = MACRO_DIM,
            lob_dim     = LOB_DIM,
            priv_dim    = PRIV_DIM,
            gru_hidden  = GRU_HIDDEN,
            macro_embed = MACRO_EMBED,
            fc_hidden   = FC_HIDDEN,
            n_dir       = N_DIR,
            n_size      = N_SIZE,
        )
        model.load_state_dict(state_dict)
        model = model.to(DEVICE)
    except Exception as exc:
        print(f'{ticker}: failed to load model — {exc}')
        continue

    # ── Run backtest on val slice ─────────────────────────────────────────────
    res = run_backtest(
        model          = model,
        macro_feats    = macro_feats[split_bar:],
        lob_feats      = lob_feats[split_bar:],
        close_arr      = close_arr[split_bar:],
        val_day_starts = val_day_starts,
        lookback       = LOOKBACK_BARS,
        tc_pct         = TC_PCT,
        device         = DEVICE,
    )

    all_results[ticker]   = res
    equity_curves[ticker] = res['equity_curve']

    print(
        f'{ticker:8s}  Return={res["total_return"]*100:+7.2f}%  '
        f'Sharpe={res["sharpe"]:+6.3f}  '
        f'MaxDD={res["max_drawdown"]*100:6.2f}%  '
        f'WinRate={res["win_rate"]*100:5.1f}%  '
        f'Trades={res["n_trades"]}'
    )

print(f'\nBacktest complete: {len(all_results)} / {len(SP100_TICKERS)} tickers processed.')

In [ ]:
if not all_results:
    print('No results to display. Run the backtest cell first.')
else:
    # ── Metrics table ─────────────────────────────────────────────────────────
    records = [
        {
            'Ticker':          t,
            'Return (%)':      round(v['total_return'] * 100, 2),
            'Sharpe':          round(v['sharpe'],             3),
            'Max Drawdown (%)': round(v['max_drawdown'] * 100,  2),
            'Win Rate (%)':    round(v['win_rate'] * 100,      1),
            'Trades':          v['n_trades'],
        }
        for t, v in all_results.items()
    ]
    df = pd.DataFrame(records).sort_values('Sharpe', ascending=False)

    print('=' * 70)
    print('  DEEPSCALPER BACKTEST PERFORMANCE REPORT (held-out val set)')
    print('=' * 70)
    print(df.to_string(index=False))
    print('=' * 70)
    print(f"  Tickers traded : {len(df)}")
    print(f"  Median Sharpe  : {df['Sharpe'].median():.3f}")
    print(f"  Median Return  : {df['Return (%)'].median():.2f}%")
    print(f"  Positive Sharpe: {(df['Sharpe'] > 0).sum()} / {len(df)}")
    print('=' * 70)

    # ── Equity curve plot ─────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(14, 5), facecolor='#0d1117')
    ax.set_facecolor('#0d1117')

    # Individual ticker curves (faint)
    for ticker, curve in equity_curves.items():
        color = '#00d4aa' if all_results[ticker]['total_return'] >= 0 else '#ff6b6b'
        ax.plot(curve, color=color, linewidth=0.4, alpha=0.15)

    # Equal-weight aggregate curve
    if len(equity_curves) > 1:
        min_len = min(len(c) for c in equity_curves.values())
        agg = np.array([c[:min_len] for c in equity_curves.values()]).mean(axis=0)
        ax.plot(agg, color='#00d4aa', linewidth=2.0, label=f'Equal-weight avg ({len(equity_curves)} tickers)')

    ax.axhline(1.0, color='#8b949e', linestyle='--', linewidth=0.8, label='Breakeven (1.0×)')

    ax.set_title('DeepScalper — Held-Out Validation Equity Curves (1-min bars)',
                 color='white', fontsize=13)
    ax.set_xlabel('Bar step', color='#8b949e')
    ax.set_ylabel('Normalised portfolio value', color='#8b949e')
    ax.tick_params(colors='#8b949e')
    ax.legend(facecolor='#161b22', labelcolor='white', fontsize=9)
    for spine in ax.spines.values():
        spine.set_color('#30363d')

    plt.tight_layout()
    out_path = '/content/equity_curve.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'Equity curve saved → {out_path}')